# RQ7: Practical Usefulness and Final Recommendation

This Kaggle-ready notebook uses **classification** with `Severity Level` as the target variable.

Input path used first:

`/kaggle/input/datasets/teamincribo/cyber-security-attacks`

The notebook also falls back to recursive search under `/kaggle/input`.

All tables are saved as CSV files and all figures are saved as PDF files.

Leakage-aware default: `Anomaly Scores`, `Action Taken`, and `Attack Type` are removed from the feature set.

## Common setup and data preparation

In [ ]:
import os
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix
)
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.naive_bayes import BernoulliNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TARGET_COLUMN = "Severity Level"
PREFERRED_INPUT_DIR = Path("/kaggle/input/datasets/teamincribo/cyber-security-attacks")
OUTPUT_ROOT = Path("/kaggle/working/severity_classification_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Output directory:", OUTPUT_ROOT)

def find_dataset_file(preferred_dir=PREFERRED_INPUT_DIR):
    supported_ext = [".csv", ".xlsx", ".xls"]
    roots = []
    if Path(preferred_dir).exists():
        roots.append(Path(preferred_dir))
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    for r in [Path("."), Path("/mnt/data")]:
        if r.exists():
            roots.append(r)

    candidates = []
    for root in roots:
        for ext in supported_ext:
            candidates.extend(root.rglob(f"*{ext}"))

    candidates = [c for c in candidates if not any(x in c.name.lower() for x in ["sample_submission", "submission"])]

    if not candidates:
        raise FileNotFoundError("No CSV/XLS/XLSX dataset file found.")

    keywords = ["cyber", "security", "attack"]
    candidates = sorted(
        candidates,
        key=lambda p: (str(p).startswith(str(PREFERRED_INPUT_DIR)), sum(k in p.name.lower() for k in keywords)),
        reverse=True
    )
    print("Selected dataset file:", candidates[0])
    return candidates[0]

def load_raw_dataset():
    path = find_dataset_file()
    if path.suffix.lower() == ".csv":
        data = pd.read_csv(path)
    else:
        data = pd.read_excel(path)
    print("Dataset shape:", data.shape)
    display(data.head())
    return data

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True, min_frequency=20)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

def prepare_classification_features(
    data,
    include_attack_type=False,
    include_anomaly_scores=False,
    create_missing_indicators=True,
    use_timestamp_features=True,
    use_ip_prefix_features=True,
    drop_high_cardinality_text=True
):
    if TARGET_COLUMN not in data.columns:
        raise KeyError(f"Target column '{TARGET_COLUMN}' not found.")

    df = data.copy()
    y = df[TARGET_COLUMN].astype(str)
    X = df.drop(columns=[TARGET_COLUMN])

    leakage_cols = ["Action Taken"]
    if not include_anomaly_scores:
        leakage_cols.append("Anomaly Scores")
    if not include_attack_type:
        leakage_cols.append("Attack Type")
    X = X.drop(columns=leakage_cols, errors="ignore")

    if use_timestamp_features and "Timestamp" in X.columns:
        ts = pd.to_datetime(X["Timestamp"], errors="coerce")
        X["hour"] = ts.dt.hour
        X["dayofweek"] = ts.dt.dayofweek
        X["month"] = ts.dt.month
        X = X.drop(columns=["Timestamp"], errors="ignore")

    if use_ip_prefix_features:
        ip_map = {
            "Source IP Address": "source_ip_prefix",
            "Destination IP Address": "destination_ip_prefix",
            "Proxy Information": "proxy_ip_prefix"
        }
        for old, new in ip_map.items():
            if old in X.columns:
                X[new] = X[old].astype(str).str.split(".").str[0].replace({"nan": np.nan})
                X = X.drop(columns=[old], errors="ignore")

    if create_missing_indicators:
        for col in ["Alerts/Warnings", "IDS/IPS Alerts", "Malware Indicators", "Firewall Logs", "Proxy Information"]:
            if col in X.columns:
                X[col + "_missing"] = X[col].isna().astype(int)

    if drop_high_cardinality_text:
        X = X.drop(columns=["Payload Data", "User Information", "Device Information", "Geo-location Data"], errors="ignore")

    return X, y

def build_preprocessor(X, scale_numeric=True):
    num_cols = X.select_dtypes(include=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=["int64", "float64", "int32", "float32", "bool"]).columns.tolist()

    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler(with_mean=False)))

    preprocessor = ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("encoder", make_onehot_encoder())
        ]), cat_cols)
    ])

    return preprocessor, num_cols, cat_cols

def split_data(X, y, test_size=0.20):
    return train_test_split(X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y)

def evaluate_predictions(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision Macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Recall Macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 Macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "F1 Weighted": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

def fit_evaluate_model(name, estimator, X_train, X_test, y_train, y_test, preprocessor):
    pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
    start = time.time()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    elapsed = time.time() - start
    metrics = evaluate_predictions(y_test, pred)
    metrics["Model"] = name
    metrics["Training Time Seconds"] = elapsed
    return metrics, pred, pipe

def get_candidate_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=500, class_weight="balanced", solver="saga", n_jobs=-1, random_state=RANDOM_STATE),
        "Decision Tree Classifier": DecisionTreeClassifier(max_depth=14, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE),
        "Random Forest Classifier": RandomForestClassifier(n_estimators=20, max_depth=10, min_samples_leaf=5, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1),
        "Extra Trees Classifier": ExtraTreesClassifier(n_estimators=20, max_depth=10, min_samples_leaf=5, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
        "SGD Linear Classifier": SGDClassifier(loss="log_loss", penalty="l2", alpha=0.0005, class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE, n_jobs=-1),
        "Bernoulli Naive Bayes": BernoulliNB()
    }

def save_table(table, filename, output_dir=OUTPUT_ROOT):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    table.to_csv(path, index=False)
    print("Saved table:", path)
    return path

def save_figure(fig, filename, output_dir=OUTPUT_ROOT):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    path = output_dir / filename
    fig.savefig(path, format="pdf", bbox_inches="tight")
    plt.show()
    print("Saved figure:", path)
    return path

def style_axes(ax):
    ax.grid(True, alpha=0.30, linestyle="--")
    ax.set_axisbelow(True)

df = load_raw_dataset()

target_distribution = df[TARGET_COLUMN].value_counts(dropna=False).reset_index()
target_distribution.columns = [TARGET_COLUMN, "Count"]
display(target_distribution)

X, y = prepare_classification_features(df)
preprocessor, numeric_features, categorical_features = build_preprocessor(X, scale_numeric=True)
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.20)

print("Prepared feature matrix shape:", X.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Class labels:", sorted(y.unique()))

## RQ7: Practical Usefulness and Final Recommendation

In [ ]:
rq7_dir = OUTPUT_ROOT / "rq7_final_recommendation"
rq7_dir.mkdir(parents=True, exist_ok=True)

try:
    rq2_table
except NameError:
    results = []
    for name, model in get_candidate_models().items():
        print(f"Training {name}...")
        metrics, _, _ = fit_evaluate_model(name, model, X_train, X_test, y_train, y_test, preprocessor)
        results.append(metrics)
    rq2_table = pd.DataFrame(results)

decision_df = rq2_table.copy()
interpretability_scores = {
    "Logistic Regression": 5.0,
    "Decision Tree Classifier": 4.2,
    "Random Forest Classifier": 3.8,
    "Extra Trees Classifier": 3.4,
    "SGD Linear Classifier": 3.6,
    "Bernoulli Naive Bayes": 4.0
}
decision_df["Predictive Performance Score"] = decision_df["F1 Macro"].rank(ascending=True, method="average") / len(decision_df) * 5
decision_df["Interpretability Score"] = decision_df["Model"].map(interpretability_scores).fillna(3.0)
decision_df["Computational Cost Score"] = decision_df["Training Time Seconds"].rank(ascending=False, method="average") / len(decision_df) * 5
decision_df["Robustness Score"] = (0.5 * decision_df["Balanced Accuracy"] + 0.5 * decision_df["F1 Weighted"]).rank(ascending=True, method="average") / len(decision_df) * 5
decision_df["Deployment Suitability Score"] = 0.40 * decision_df["Predictive Performance Score"] + 0.25 * decision_df["Interpretability Score"] + 0.20 * decision_df["Robustness Score"] + 0.15 * decision_df["Computational Cost Score"]

rq7_table = decision_df[["Model", "Accuracy", "Balanced Accuracy", "F1 Macro", "Predictive Performance Score", "Interpretability Score", "Robustness Score", "Computational Cost Score", "Deployment Suitability Score"]].sort_values("Deployment Suitability Score", ascending=False)
rq7_table["Final Rank"] = range(1, len(rq7_table) + 1)
display(rq7_table)
save_table(rq7_table, "table_rq7_final_decision_matrix.csv", rq7_dir)

fig, ax = plt.subplots(figsize=(10, 7))
sizes = rq7_table["Robustness Score"] * 180
ax.scatter(rq7_table["Predictive Performance Score"], rq7_table["Interpretability Score"], s=sizes, alpha=0.65)
for _, row in rq7_table.iterrows():
    ax.text(row["Predictive Performance Score"] + 0.04, row["Interpretability Score"] + 0.04, row["Model"], fontsize=9)

recommended = rq7_table.iloc[0]
ax.scatter([recommended["Predictive Performance Score"]], [recommended["Interpretability Score"]], s=[recommended["Robustness Score"] * 230], facecolors="none", edgecolors="black", linewidths=2)
ax.annotate(f"Recommended: {recommended['Model']}", xy=(recommended["Predictive Performance Score"], recommended["Interpretability Score"]), xytext=(0.50, 0.15), textcoords="axes fraction", arrowprops=dict(arrowstyle="->", lw=1.5))

ax.set_title("Figure 7. Final Trade-off Analysis for Severity Level Classifier Selection")
ax.set_xlabel("Predictive Performance Score")
ax.set_ylabel("Interpretability Score")
ax.set_xlim(0.5, 5.5)
ax.set_ylim(0.5, 5.5)
style_axes(ax)
ax.text(0.02, 0.02, "Bubble size represents robustness score", transform=ax.transAxes, fontsize=9)
fig.tight_layout()
save_figure(fig, "figure_rq7_final_tradeoff.pdf", rq7_dir)

print("RQ7 answer: Recommended practical model =", recommended["Model"])

## Output inventory

In [ ]:
all_outputs = []
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file() and path.suffix.lower() in [".csv", ".pdf"]:
        all_outputs.append({"File": path.name, "Path": str(path), "Type": path.suffix.lower()})
inventory_df = pd.DataFrame(all_outputs)
display(inventory_df)
save_table(inventory_df, "output_inventory.csv", OUTPUT_ROOT)
print("All outputs were saved under:", OUTPUT_ROOT)